### Task 10: Model Evaluation and Optimization

#### Objective

The objective of this task is to evaluate and optimize the performance
of Content-Based Filtering, Collaborative Filtering, and Hybrid
Recommendation models.

The models will be evaluated using recommendation metrics such as
Precision@K and Recall@K. Different hybrid weights will also be tested
to identify the most suitable model configuration.

#### Evaluation Techniques

1. Content-Based Filtering
2. User-Based Collaborative Filtering
3. Hybrid Recommendation Model
4. Precision@K
5. Recall@K

#### Dataset

The cleaned TED Talks dataset and simulated user interaction dataset
created in previous tasks will be used.

Note: User interaction data is simulated for educational and
demonstration purposes.

In [1]:
import pandas as pd
import numpy as np

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# Load datasets
ted_df = pd.read_csv("../data/processed/ted_talks_cleaned.csv")
interactions_df = pd.read_csv("../data/interaction/user_interactions.csv")

print("TED Talks Dataset Shape:", ted_df.shape)
print("User Interactions Dataset Shape:", interactions_df.shape)

display(ted_df.head())
display(interactions_df.head())

TED Talks Dataset Shape: (4005, 26)
User Interactions Dataset Shape: (37, 7)


,talk_id,title,speaker_1,all_speakers,occupations,about_speakers,views,recorded_date,published_date,event,...,url,description,transcript,title_clean,description_clean,transcript_clean,topics_clean,combined_text,combined_text_length,combined_text_word_count
0,1,Averting the climate crisis,Al Gore,{0: 'Al Gore'},{0: ['climate advocate']},{0: 'Nobel Laureate Al Gore focused the world’...,3523392,2006-02-25,2006-06-27,TED2006,...,https://www.ted.com/talks/al_gore_averting_the...,With the same humor and humanity he exuded in ...,"Thank you so much, Chris. And it's truly a gre...",averting the climate crisis,with the same humor and humanity he exuded in ...,thank you so much chris and it s truly a great...,alternative energy cars climate change culture...,averting the climate crisis alternative energy...,11819,2269
1,92,The best stats you've ever seen,Hans Rosling,{0: 'Hans Rosling'},{0: ['global health expert; data visionary']},"{0: 'In Hans Rosling’s hands, data sings. Glob...",14501685,2006-02-22,2006-06-27,TED2006,...,https://www.ted.com/talks/hans_rosling_the_bes...,You've never seen data presented like this. Wi...,"About 10 years ago, I took on the task to teac...",the best stats you ve ever seen,you ve never seen data presented like this wit...,about 10 years ago i took on the task to teach...,africa asia google demo economics global issue...,the best stats you ve ever seen africa asia go...,17185,3278
2,7,Simplicity sells,David Pogue,{0: 'David Pogue'},{0: ['technology columnist']},{0: 'David Pogue is the personal technology co...,1920832,2006-02-24,2006-06-27,TED2006,...,https://www.ted.com/talks/david_pogue_simplici...,New York Times columnist David Pogue takes aim...,"(Music: ""The Sound of Silence,"" Simon & Garfun...",simplicity sells,new york times columnist david pogue takes aim...,music the sound of silence simon garfunkel hel...,computers entertainment interface design media...,simplicity sells computers entertainment inter...,17892,3548
3,53,Greening the ghetto,Majora Carter,{0: 'Majora Carter'},{0: ['activist for environmental justice']},{0: 'Majora Carter redefined the field of envi...,2664069,2006-02-26,2006-06-27,TED2006,...,https://www.ted.com/talks/majora_carter_greeni...,"In an emotionally charged talk, MacArthur-winn...",If you're here today — and I'm very happy that...,greening the ghetto,in an emotionally charged talk macarthur winni...,if you re here today and i m very happy that y...,macarthur grant activism business cities envir...,greening the ghetto macarthur grant activism b...,18150,3226
4,66,Do schools kill creativity?,Sir Ken Robinson,{0: 'Sir Ken Robinson'},"{0: ['author', 'educator']}","{0: ""Creativity expert Sir Ken Robinson challe...",65051954,2006-02-25,2006-06-27,TED2006,...,https://www.ted.com/talks/sir_ken_robinson_do_...,Sir Ken Robinson makes an entertaining and pro...,Good morning. How are you? (Audience) Good. It...,do schools kill creativity,sir ken robinson makes an entertaining and pro...,good morning how are you audience good it s be...,children creativity culture dance education pa...,do schools kill creativity children creativity...,17061,3326


,user_id,talk_id,title,rating,liked,watched,bookmarked
0,U001,1,Averting the climate crisis,5,True,True,True
1,U001,92,The best stats you've ever seen,4,True,True,False
2,U001,7,Simplicity sells,3,False,True,False
3,U001,53,Greening the ghetto,5,True,True,True
4,U001,66,Do schools kill creativity?,2,False,True,False


### 3. Preparing Training and Testing Data

The interaction dataset is divided into training and testing sets.

One positively rated TED Talk is held out for each user.
The models use the remaining interactions to generate recommendations.

A rating of 4 or higher is considered relevant.

A fixed random seed is used to make the evaluation reproducible.

In [2]:
# Select positively rated interactions
positive_interactions = interactions_df[
    interactions_df["rating"] >= 4
]

# Select one positive interaction for each user
test_indices = []

for user in positive_interactions["user_id"].unique():
    user_positive = positive_interactions[
        positive_interactions["user_id"] == user
    ]

    selected = user_positive.sample(n=1, random_state=42)
    test_indices.extend(selected.index)

# Create training and testing datasets
train_interactions = interactions_df.drop(
    index=test_indices
).copy()

test_interactions = interactions_df.loc[
    test_indices
].copy()

print("Training Dataset Shape:", train_interactions.shape)
print("Testing Dataset Shape:", test_interactions.shape)

print("\nHeld-out Relevant Talks:")
display(
    test_interactions[
        ["user_id", "talk_id", "title", "rating"]
    ].sort_values("user_id")
)

Training Dataset Shape: (32, 7)
Testing Dataset Shape: (5, 7)

Held-out Relevant Talks:


,user_id,talk_id,title,rating
1,U001,92,The best stats you've ever seen,4
9,U002,92,The best stats you've ever seen,5
18,U003,53,Greening the ghetto,4
24,U004,92,The best stats you've ever seen,4
31,U005,7,Simplicity sells,4


### 4. Building Models Using Training Data

The recommendation models are rebuilt using only the training interactions.

The Content-Based model uses TED Talk titles, descriptions, and topics.
The User-Item Matrix is created using the available training ratings.

In [3]:
# Create User-Item Matrix using training data
user_item_matrix = train_interactions.pivot_table(
    index="user_id",
    columns="talk_id",
    values="rating"
)

print("Training User-Item Matrix:")
display(user_item_matrix)

# Prepare content features
content_df = ted_df[
    ["talk_id", "title", "title_clean",
     "description_clean", "topics_clean"]
].copy()

content_features = [
    "title_clean",
    "description_clean",
    "topics_clean"
]

content_df[content_features] = content_df[
    content_features
].fillna("")

content_df = content_df.drop_duplicates(
    subset="talk_id"
).reset_index(drop=True)

# Combine content features
content_df["content_text"] = (
    content_df["title_clean"] + " " +
    content_df["description_clean"] + " " +
    content_df["topics_clean"]
)

# Create TF-IDF vectors
tfidf = TfidfVectorizer(
    stop_words="english",
    max_features=5000
)

content_vectors = tfidf.fit_transform(
    content_df["content_text"]
)

# Calculate content similarity
content_similarity = cosine_similarity(content_vectors)

content_similarity_df = pd.DataFrame(
    content_similarity,
    index=content_df["talk_id"],
    columns=content_df["talk_id"]
)

print("\nContent-Based Model Created Successfully!")
print("Content Similarity Matrix Shape:", content_similarity.shape)

Training User-Item Matrix:


talk_id,1,7,49,53,66,86,92,94
user_id,,,,,,,,
U001,5.0,3.0,4.0,5.0,2.0,5.0,NaN,3.0
U002,4.0,2.0,3.0,4.0,5.0,4.0,NaN,NaN
U003,2.0,5.0,5.0,NaN,3.0,4.0,3.0,2.0
U004,5.0,4.0,4.0,2.0,5.0,NaN,NaN,NaN
U005,3.0,NaN,2.0,5.0,3.0,4.0,5.0,5.0



Content-Based Model Created Successfully!
Content Similarity Matrix Shape: (4005, 4005)


### 5. Building the Collaborative Filtering Model

User-Based Collaborative Filtering identifies users with similar rating
patterns and predicts ratings for TED Talks that the target user
has not rated.

Cosine Similarity is used to calculate similarities between users.
Only training interactions are used to build this model.

In [4]:
# Fill missing ratings with zero for similarity calculation
user_item_filled = user_item_matrix.fillna(0)

# Calculate user similarity
user_similarity = cosine_similarity(user_item_filled)

# Create similarity DataFrame
user_similarity_df = pd.DataFrame(
    user_similarity,
    index=user_item_matrix.index,
    columns=user_item_matrix.index
)

print("Collaborative Filtering Model Created Successfully!")

print("\nUser Similarity Matrix:")
display(user_similarity_df.round(3))

Collaborative Filtering Model Created Successfully!

User Similarity Matrix:


user_id,U001,U002,U003,U004,U005
user_id,,,,,
U001,1.000,0.893,0.755,0.741,0.788
U002,0.893,1.000,0.720,0.849,0.700
U003,0.755,0.720,1.000,0.731,0.647
U004,0.741,0.849,0.731,1.000,0.487
U005,0.788,0.700,0.647,0.487,1.000


### 6. Creating Recommendation Functions

Three recommendation functions are developed:

1. Content-Based Recommendation
2. Collaborative Filtering Recommendation
3. Hybrid Recommendation

All three functions use only the training data to generate recommendations.
The held-out test interactions will be used later for evaluation.

In [5]:

# 1. Content-Based Recommendation Function

def get_content_recommendations(user_id, top_n=5):

    if user_id not in user_item_matrix.index:
        return pd.DataFrame(
            columns=["talk_id", "content_score"]
        )

    # Find talks liked by the user in training data
    liked_talks = user_item_matrix.loc[user_id]
    liked_talks = liked_talks[liked_talks >= 4].index

    if len(liked_talks) == 0:
        return pd.DataFrame(
            columns=["talk_id", "content_score"]
        )

    # Exclude talks already rated during training
    rated_talks = user_item_matrix.loc[user_id].dropna().index

    # Calculate maximum similarity to liked talks
    similarity_scores = content_similarity_df.loc[
        :, liked_talks
    ].max(axis=1)

    similarity_scores = similarity_scores.drop(
        labels=rated_talks,
        errors="ignore"
    )

    recommendations = similarity_scores.sort_values(
        ascending=False
    ).head(top_n)

    return recommendations.rename(
        "content_score"
    ).reset_index().rename(
        columns={"index": "talk_id"}
    )


# 2. Collaborative Filtering Recommendation Function

def get_collaborative_recommendations(user_id, top_n=5):

    if user_id not in user_item_matrix.index:
        return pd.DataFrame(
            columns=["talk_id", "collaborative_score"]
        )

    user_ratings = user_item_matrix.loc[user_id]

    # Find talks not rated by the target user
    unrated_talks = user_ratings[user_ratings.isna()].index

    # Find other users in descending similarity order
    similar_users = (
        user_similarity_df[user_id]
        .drop(user_id)
        .sort_values(ascending=False)
    )

    recommendations = []

    for talk_id in unrated_talks:

        ratings = user_item_matrix[talk_id].dropna()

        common_users = similar_users.index.intersection(
            ratings.index
        )

        if len(common_users) == 0:
            continue

        weights = similar_users.loc[common_users]
        user_ratings_for_talk = ratings.loc[common_users]

        if weights.sum() <= 0:
            continue

        predicted_rating = (
            weights * user_ratings_for_talk
        ).sum() / weights.sum()

        recommendations.append({
            "talk_id": talk_id,
            "collaborative_score": predicted_rating
        })

    recommendations_df = pd.DataFrame(recommendations)

    if recommendations_df.empty:
        return pd.DataFrame(
            columns=["talk_id", "collaborative_score"]
        )

    return recommendations_df.sort_values(
        "collaborative_score",
        ascending=False
    ).head(top_n).reset_index(drop=True)


# 3. Hybrid Recommendation Function

def hybrid_recommend(
    user_id,
    top_n=5,
    content_weight=0.5,
    collaborative_weight=0.5
):

    if not np.isclose(
        content_weight + collaborative_weight, 1
    ) or content_weight < 0 or collaborative_weight < 0:
        raise ValueError(
            "Weights must be non-negative and add up to 1."
        )

    content_results = get_content_recommendations(
        user_id, top_n
    )

    collaborative_results = get_collaborative_recommendations(
        user_id, top_n
    )

    hybrid_df = pd.merge(
        content_results,
        collaborative_results,
        on="talk_id",
        how="outer"
    )

    if hybrid_df.empty:
        return pd.DataFrame(
            columns=[
                "talk_id",
                "content_score",
                "collaborative_score",
                "hybrid_score"
            ]
        )

    # Normalize collaborative ratings from 1-5 to 0-1
    hybrid_df["collaborative_score"] = (
        hybrid_df["collaborative_score"].fillna(1) - 1
    ) / 4

    hybrid_df["content_score"] = (
        hybrid_df["content_score"].fillna(0)
    )

    # Calculate weighted hybrid score
    hybrid_df["hybrid_score"] = (
        content_weight * hybrid_df["content_score"]
        + collaborative_weight * hybrid_df["collaborative_score"]
    )

    return (
        hybrid_df.drop_duplicates(subset="talk_id")
        .sort_values("hybrid_score", ascending=False)
        .head(top_n)
        .reset_index(drop=True)
    )


print("All Three Recommendation Functions Created Successfully!")


All Three Recommendation Functions Created Successfully!


### 7. Generating Recommendations for Evaluation

Content-Based, Collaborative Filtering, and Hybrid models are used
to generate the top 5 recommendations for each user.

The recommendations are generated using training data only.
The hidden test interactions are kept separate for evaluation.

In [6]:

# Create a lookup table for TED Talk titles
talk_lookup = ted_df[
    ["talk_id", "title"]
].drop_duplicates(subset="talk_id")

# Store recommendations for all users
evaluation_users = sorted(train_interactions["user_id"].unique())

for user in evaluation_users:

    print(f"\n{'=' * 60}")
    print(f"RECOMMENDATIONS FOR {user}")
    print(f"{'=' * 60}")

    # Content-Based recommendations
    content_results = get_content_recommendations(user, top_n=5)
    content_results = content_results.merge(
        talk_lookup, on="talk_id", how="left"
    )

    print("\nContent-Based Recommendations:")
    display(content_results)

    # Collaborative recommendations
    collaborative_results = get_collaborative_recommendations(
        user, top_n=5
    )
    collaborative_results = collaborative_results.merge(
        talk_lookup, on="talk_id", how="left"
    )

    print("\nCollaborative Filtering Recommendations:")
    display(collaborative_results)

    # Hybrid recommendations
    hybrid_results = hybrid_recommend(user, top_n=5)
    hybrid_results = hybrid_results.merge(
        talk_lookup, on="talk_id", how="left"
    )

    print("\nHybrid Recommendations:")
    display(hybrid_results)



RECOMMENDATIONS FOR U001

Content-Based Recommendations:


,talk_id,content_score,title
0,243,0.391374,New thinking on the climate crisis
1,535,0.373878,What comes after An Inconvenient Truth?
2,418,0.357973,My library of human imagination
3,2441,0.342382,The case for optimism on climate change
4,51,0.331965,Winning the oil endgame



Collaborative Filtering Recommendations:


,talk_id,collaborative_score,title
0,92,4.021013,The best stats you've ever seen



Hybrid Recommendations:


,talk_id,content_score,collaborative_score,hybrid_score,title
0,92,0.000000,0.755253,0.377627,The best stats you've ever seen
1,243,0.391374,0.000000,0.195687,New thinking on the climate crisis
2,535,0.373878,0.000000,0.186939,What comes after An Inconvenient Truth?
3,418,0.357973,0.000000,0.178986,My library of human imagination
4,2441,0.342382,0.000000,0.171191,The case for optimism on climate change


... (20 outputs left out)


Collaborative Filtering Recommendations:


,talk_id,collaborative_score,title
0,7,3.412542,Simplicity sells



Hybrid Recommendations:


,talk_id,content_score,collaborative_score,hybrid_score,title
0,7,0.000000,0.603135,0.301568,Simplicity sells
1,1327,0.451540,0.000000,0.225770,Atheism 2.0
2,2011,0.400438,0.000000,0.200219,Is religion good or bad? (This is a trick ques...
3,620,0.380255,0.000000,0.190127,Let my dataset change your mindset
4,140,0.337244,0.000000,0.168622,New insights on poverty


### 8. Model Evaluation Using Precision@K and Recall@K

Precision@5 measures the proportion of relevant talks among the
top five recommendations.

Recall@5 measures the proportion of held-out relevant talks
that appear in the top five recommendations.

A rating of 4 or higher is considered relevant.

In [7]:

# Set evaluation parameter
K = 5

# Store evaluation results
evaluation_results = []

# Define recommendation models
recommendation_models = {
    "Content-Based": get_content_recommendations,
    "Collaborative Filtering": get_collaborative_recommendations,
    "Hybrid": hybrid_recommend
}

# Evaluate each model
for model_name, recommendation_function in recommendation_models.items():

    user_precisions = []
    user_recalls = []

    for user in sorted(test_interactions["user_id"].unique()):

        # Get relevant held-out talks for this user
        relevant_talks = set(
            test_interactions[
                (test_interactions["user_id"] == user) &
                (test_interactions["rating"] >= 4)
            ]["talk_id"]
        )

        # Generate top K recommendations using training data
        recommendations = recommendation_function(user, top_n=K)

        recommended_talks = set(
            recommendations["talk_id"].head(K)
        )

        # Calculate relevant recommended talks
        hits = len(relevant_talks.intersection(recommended_talks))

        # Calculate Precision@K and Recall@K
        precision = hits / K if K > 0 else 0

        recall = (
            hits / len(relevant_talks)
            if len(relevant_talks) > 0 else 0
        )

        user_precisions.append(precision)
        user_recalls.append(recall)

    # Calculate average metrics across users
    evaluation_results.append({
        "Model": model_name,
        "Precision@5": np.mean(user_precisions),
        "Recall@5": np.mean(user_recalls)
    })

# Create evaluation comparison table
evaluation_df = pd.DataFrame(evaluation_results)

print("Model Evaluation Results:")
display(evaluation_df.round(3))


Model Evaluation Results:


,Model,Precision@5,Recall@5
0,Content-Based,0.0,0.0
1,Collaborative Filtering,0.2,1.0
2,Hybrid,0.2,1.0


#### Understand the evaluation results
Interpretation:
1. Content-Based: None of the held-out relevant talks appeared in the top five recommendations.
2. Collaborative Filtering: It successfully recommended the held-out relevant talks.
3. Hybrid: It achieved the same evaluation scores as Collaborative Filtering with equal weights.
Since we held out one relevant talk per user, Recall@5 of 1.0 means all five held-out relevant talks were successfully recommended across the five users.

Precision@5 of 0.20 means, on average, one out of five recommendations was relevant.

Important: These results are based on only five simulated test interactions. We should not conclude that Collaborative Filtering will always outperform Content-Based Filtering.

### 9. Hybrid Model Weight Optimization

Different combinations of Content-Based and Collaborative Filtering
weights are tested to identify the most suitable hybrid configuration.

The performance of each configuration is evaluated using Precision@5
and Recall@5.

In [8]:

# Define different weight combinations
weight_combinations = [
    (1.00, 0.00),
    (0.75, 0.25),
    (0.50, 0.50),
    (0.25, 0.75),
    (0.00, 1.00)
]

weight_results = []

# Test each weight combination
for content_weight, collaborative_weight in weight_combinations:

    user_precisions = []
    user_recalls = []

    for user in sorted(test_interactions["user_id"].unique()):

        # Get relevant held-out talks
        relevant_talks = set(
            test_interactions[
                (test_interactions["user_id"] == user) &
                (test_interactions["rating"] >= 4)
            ]["talk_id"]
        )

        # Generate hybrid recommendations
        recommendations = hybrid_recommend(
            user,
            top_n=5,
            content_weight=content_weight,
            collaborative_weight=collaborative_weight
        )

        recommended_talks = set(
            recommendations["talk_id"].head(5)
        )

        hits = len(relevant_talks.intersection(recommended_talks))

        precision = hits / 5
        recall = hits / len(relevant_talks) if relevant_talks else 0

        user_precisions.append(precision)
        user_recalls.append(recall)

    weight_results.append({
        "Content Weight": content_weight,
        "Collaborative Weight": collaborative_weight,
        "Precision@5": np.mean(user_precisions),
        "Recall@5": np.mean(user_recalls)
    })

# Display weight comparison
weight_evaluation_df = pd.DataFrame(weight_results)

print("Hybrid Weight Optimization Results:")
display(weight_evaluation_df.round(3))


Hybrid Weight Optimization Results:


,Content Weight,Collaborative Weight,Precision@5,Recall@5
0,1.00,0.00,0.0,0.0
1,0.75,0.25,0.0,0.0
2,0.50,0.50,0.2,1.0
3,0.25,0.75,0.2,1.0
4,0.00,1.00,0.2,1.0


### 10. Creating Final Model Performance Comparison Table
The performance of Content-Based Filtering, Collaborative Filtering,
and the Hybrid Recommendation Model is compared using Precision@5
and Recall@5.

The Hybrid Model uses the selected configuration of 50% Content-Based
Filtering and 50% Collaborative Filtering.

In [9]:
final_comparison_df = pd.DataFrame([
    {
        "Model": "Content-Based Filtering",
        "Precision@5": 0.0,
        "Recall@5": 0.0
    },
    {
        "Model": "Collaborative Filtering",
        "Precision@5": 0.2,
        "Recall@5": 1.0
    },
    {
        "Model": "Hybrid Model (50:50)",
        "Precision@5": 0.2,
        "Recall@5": 1.0
    }
])

display(final_comparison_df)

,Model,Precision@5,Recall@5
0,Content-Based Filtering,0.0,0.0
1,Collaborative Filtering,0.2,1.0
2,Hybrid Model (50:50),0.2,1.0


### 11. Strengths, Weaknesses, and Limitations

#### 1. Content-Based Filtering

**Strengths:**
- Recommends TED Talks based on content similarity.
- Does not depend on other users' ratings.
- Can recommend talks with no previous user interactions if their content information is available.

**Weaknesses:**
- Recommendations depend heavily on the quality of descriptions, titles, and tags.
- May recommend similar topics repeatedly.
- Requires some user preferences to generate personalized recommendations.

#### 2. Collaborative Filtering

**Strengths:**
- Identifies user preferences through interaction patterns.
- Can discover talks that are not necessarily similar in content.
- Achieved Precision@5 of 0.2 and Recall@5 of 1.0 in this experiment.

**Weaknesses:**
- Depends on sufficient user interaction data.
- Cannot effectively recommend items with no recorded interactions.
- Performance can decrease when the interaction matrix is sparse.

#### 3. Hybrid Recommendation Model

**Strengths:**
- Combines content similarity and collaborative filtering.
- Uses information from both talk features and user interactions.
- Provides a balance between the two recommendation approaches.

**Weaknesses:**
- Requires additional computation and model integration.
- Performance depends on the selected weights.
- Limited interaction data can affect recommendation quality.

#### 4. Overall Limitations

- The interaction dataset contains only five simulated users.
- The evaluation uses only five held-out positive interactions.
- The evaluation results may change with a larger or different dataset.
- No repeated train-test splits or cross-validation were performed.
- Cold-start problems can affect new users and newly added TED Talks.
- The evaluation is based on simulated interactions rather than real user behaviour.

### 12. Final Model Selection

Based on the evaluation results, the Hybrid Recommendation Model
with 50% Content-Based Filtering and 50% Collaborative Filtering
is selected as the final model.

#### Reasons for Selection

1. The Hybrid Model achieved Precision@5 of 0.2 and Recall@5 of 1.0.
2. It successfully retrieved all five held-out relevant TED Talks.
3. It combines content information with user interaction patterns.
4. It provides a balanced contribution from both recommendation approaches.
5. It achieved the same evaluation scores as configurations with higher collaborative weights.

#### Final Configuration

- Content-Based Weight: 50%
- Collaborative Filtering Weight: 50%
- Evaluation Metric: Precision@5 and Recall@5
- Selected Model: Hybrid Recommendation Model

The selected configuration is suitable for this educational project
and the available simulated dataset. Further evaluation with a larger
real-world dataset would be necessary before practical deployment.

### 13. Conclusion

In this task, Content-Based Filtering, Collaborative Filtering, and
Hybrid Recommendation models were evaluated using Precision@5 and
Recall@5.

The evaluation showed that Collaborative Filtering and the Hybrid
Model achieved Precision@5 of 0.2 and Recall@5 of 1.0, while
Content-Based Filtering achieved 0.0 for both metrics.

Different hybrid weight combinations were also tested. The 50:50
configuration was selected because it achieved the best observed
performance while maintaining a balanced contribution from both models.

The Hybrid Recommendation Model was selected as the final model
for the TED Talks Recommendation System.

This task provided an understanding of recommendation evaluation,
model comparison, weight optimization, and limitations such as
sparse interaction data and cold-start problems.

The results are based on a small simulated dataset and should be
validated further using real-world user interactions.

In [11]:
#Create the PDF report

import os
os.makedirs("reports", exist_ok=True)

from reportlab.lib.pagesizes import A4
from reportlab.platypus import (
    SimpleDocTemplate, Paragraph, Spacer, Table, TableStyle
)
from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle
from reportlab.lib import colors
from reportlab.lib.units import inch

output_path = "reports/task_10_observations.pdf"

doc = SimpleDocTemplate(
    output_path,
    pagesize=A4,
    rightMargin=45,
    leftMargin=45,
    topMargin=45,
    bottomMargin=45
)

styles = getSampleStyleSheet()
styles.add(ParagraphStyle(
    name="ReportTitle",
    parent=styles["Title"],
    fontSize=19,
    leading=24,
    spaceAfter=15
))
styles.add(ParagraphStyle(
    name="SectionHeading",
    parent=styles["Heading2"],
    fontSize=13,
    leading=17,
    spaceBefore=12,
    spaceAfter=7
))
styles.add(ParagraphStyle(
    name="ReportBody",
    parent=styles["BodyText"],
    fontSize=9.5,
    leading=14,
    spaceAfter=7
))

story = []

story.append(Paragraph(
    "Task 10: Model Evaluation and Optimization",
    styles["ReportTitle"]
))

story.append(Paragraph("<b>Project:</b> Personalized TED Talks Recommendation System Using Machine Learning", styles["ReportBody"]))
story.append(Paragraph("<b>Objective</b>", styles["SectionHeading"]))
story.append(Paragraph(
    "The objective was to evaluate Content-Based Filtering, "
    "Collaborative Filtering, and Hybrid Recommendation models "
    "using Precision@5 and Recall@5, experiment with hybrid weights, "
    "and select a final model configuration.",
    styles["ReportBody"]
))

story.append(Paragraph("<b>Evaluation Methodology</b>", styles["SectionHeading"]))
story.append(Paragraph(
    "The simulated user interaction dataset was divided into training "
    "and testing sets. One positively rated interaction (rating of 4 "
    "or higher) was held out for each of the five users. The models "
    "generated the top five recommendations using training data only.",
    styles["ReportBody"]
))

story.append(Paragraph("<b>Model Performance Comparison</b>", styles["SectionHeading"]))

comparison_data = [
    ["Model", "Precision@5", "Recall@5"],
    ["Content-Based Filtering", "0.0", "0.0"],
    ["Collaborative Filtering", "0.2", "1.0"],
    ["Hybrid Model (50:50)", "0.2", "1.0"]
]

comparison_table = Table(
    comparison_data,
    colWidths=[3.0 * inch, 1.2 * inch, 1.0 * inch]
)
comparison_table.setStyle(TableStyle([
    ("BACKGROUND", (0, 0), (-1, 0), colors.HexColor("#D9EAF7")),
    ("TEXTCOLOR", (0, 0), (-1, 0), colors.black),
    ("FONTNAME", (0, 0), (-1, 0), "Helvetica-Bold"),
    ("GRID", (0, 0), (-1, -1), 0.5, colors.grey),
    ("FONTSIZE", (0, 0), (-1, -1), 9),
    ("PADDING", (0, 0), (-1, -1), 7),
    ("VALIGN", (0, 0), (-1, -1), "MIDDLE")
]))
story.append(comparison_table)

story.append(Paragraph("<b>Hybrid Weight Optimization</b>", styles["SectionHeading"]))

weight_data = [
    ["Content Weight", "Collaborative Weight", "Precision@5", "Recall@5"],
    ["100%", "0%", "0.0", "0.0"],
    ["75%", "25%", "0.0", "0.0"],
    ["50%", "50%", "0.2", "1.0"],
    ["25%", "75%", "0.2", "1.0"],
    ["0%", "100%", "0.2", "1.0"]
]

weight_table = Table(
    weight_data,
    colWidths=[1.2 * inch, 1.5 * inch, 1.1 * inch, 1.1 * inch]
)
weight_table.setStyle(TableStyle([
    ("BACKGROUND", (0, 0), (-1, 0), colors.HexColor("#D9EAF7")),
    ("FONTNAME", (0, 0), (-1, 0), "Helvetica-Bold"),
    ("GRID", (0, 0), (-1, -1), 0.5, colors.grey),
    ("FONTSIZE", (0, 0), (-1, -1), 8.5),
    ("PADDING", (0, 0), (-1, -1), 6),
    ("VALIGN", (0, 0), (-1, -1), "MIDDLE")
]))
story.append(weight_table)

story.append(Paragraph("<b>Observations</b>", styles["SectionHeading"]))
observations = [
    "Content-Based Filtering achieved Precision@5 and Recall@5 of 0.0.",
    "Collaborative Filtering achieved Precision@5 of 0.2 and Recall@5 of 1.0.",
    "The 50:50 Hybrid Model achieved the same scores as the more collaborative-heavy configurations.",
    "The Hybrid Model combines content information and user interaction patterns."
]
for item in observations:
    story.append(Paragraph(f"• {item}", styles["ReportBody"]))

story.append(Paragraph("<b>Limitations</b>", styles["SectionHeading"]))
story.append(Paragraph(
    "The dataset contains only five simulated users and five held-out "
    "positive interactions. Results may vary with a larger dataset. "
    "The experiment did not use repeated splits or cross-validation. "
    "Sparse interactions and cold-start problems may also affect "
    "recommendation quality.",
    styles["ReportBody"]
))

story.append(Paragraph("<b>Final Model Selection</b>", styles["SectionHeading"]))
story.append(Paragraph(
    "The Hybrid Recommendation Model with 50% Content-Based Filtering "
    "and 50% Collaborative Filtering was selected. It achieved "
    "Precision@5 of 0.2 and Recall@5 of 1.0 while maintaining balanced "
    "contributions from both recommendation approaches. This is the "
    "selected configuration for this educational project and requires "
    "further validation with real-world interaction data.",
    styles["ReportBody"]
))

story.append(Paragraph("<b>Conclusion</b>", styles["SectionHeading"]))
story.append(Paragraph(
    "This task demonstrated recommendation model evaluation, "
    "comparison using Precision@K and Recall@K, and hybrid weight "
    "optimization. The selected hybrid configuration provides a "
    "balanced approach for the available simulated TED Talks dataset.",
    styles["ReportBody"]
))

doc.build(story)

print("Task 10 report created successfully!")
print("Saved at:", output_path)


Task 10 report created successfully!
Saved at: reports/task_10_observations.pdf
